# Výsledky simulácie pre článok

Zozbiera výsledky bežiaceho modelu, porovná ich s číslami z DP a **uloží celú sadu**
do `simulation_for_article/<VARIANT>/`.

Ten istý notebook sa púšťa dvakrát — stačí prepnúť `VARIANT`:

| beh | VARIANT | model |
|---|---|---|
| 1. | `base_model` | pôvodný, bez zmien |
| 2. | `updated_version_1` | po úpravách z kamdojizdime |

## Predpoklady

```bash
pip install requests pandas pyarrow folium
docker run -d --name sim-base-model -p 8000:8000 simulation-brno
```

Plán a kontext: [`../simulation_for_article/PLAN.md`](../simulation_for_article/PLAN.md)

In [1]:
# ═══ NASTAVENIE ═══════════════════════════════════════════════════════
VARIANT   = "base_model"          # alebo "updated_version_1"
API_URL   = "http://localhost:8000"
POPIS     = "pôvodný model, bez zmien"   # ide do run_meta.json

SCENAR_ULICA     = "Jihlavská"    # hlavný scenár
SCENAR_NULL_LINK = 62             # hrana s ~0 objemom → odhad šumu
IBA_V_MESTE      = True           # zavrieť len úsek vnútri hranice mesta
HRANICA_MESTA    = "brno_hranica.geojson"   # OSM relácia Brno, admin_level 8
MAPA_MIN_OBJEM   = 1000           # hrany pod týmto objemom sa do máp nekreslia
# ══════════════════════════════════════════════════════════════════════

import json, subprocess, time, hashlib
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests

try:
    import folium
    HAS_FOLIUM = True
except ImportError:
    HAS_FOLIUM = False

OUT = Path("..") / "simulation_for_article" / VARIANT
OUT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

M = {}          # sem sa zbierajú všetky metriky
print(f"variant : {VARIANT}")
print(f"ukladám : {OUT.resolve()}")


variant : base_model
ukladám : /Users/magdalena.ondruskova/git/personal/Traffic-sim-backend/simulation_for_article/base_model


In [2]:
def get_json(cesta, **params):
    """GET, ktorý nespadne — vráti None a vypíše dôvod."""
    try:
        r = requests.get(f"{API_URL}{cesta}", params=params, timeout=180)
    except requests.exceptions.ConnectionError:
        print(f"  ✗ {cesta}: API nebeží")
        return None
    if r.status_code != 200:
        det = ""
        try:
            det = str(r.json().get("detail", ""))[:120]
        except Exception:
            det = r.text[:120]
        print(f"  ✗ {cesta}: {r.status_code} {det}")
        return None
    return r.json()


def geojson_to_df(gj):
    if not gj:
        return pd.DataFrame()
    rows = []
    for f in gj.get("features", []):
        p = dict(f.get("properties") or {})
        g = f.get("geometry") or {}
        t = g.get("type")
        if t == "LineString":
            p["_coords"] = g["coordinates"]
        elif t == "MultiLineString":
            p["_coords"] = [c for part in g["coordinates"] for c in part]
        else:
            p["_coords"] = None
        rows.append(p)
    return pd.DataFrame(rows)


def popis_hrany(r):
    """Popisok hrany. Diaľnice a cesty I./II. triedy nemajú v OSM `name`, len `ref`,
    takže by v tooltipe inak svietilo `nan`."""
    meno = str(r.get("name") or "").strip()
    if meno and meno.lower() != "nan":
        return meno
    ref = str(r.get("osm_ref") or "").strip()
    if ref and ref.lower() != "nan":
        prvy = ref.split(";")[0].strip()
        if prvy[:1].upper() == "D" and prvy[1:].isdigit():
            return f"dálnice {prvy}"
        return f"silnice {prvy}"
    return "(bez názvu)"


def _v_polygone(x, y, ring):
    """Ray casting — je bod vnútri prstenca? (WGS84, zámerne bez shapely)"""
    ins = False
    n = len(ring)
    for i in range(n):
        x1, y1 = ring[i][0], ring[i][1]
        x2, y2 = ring[(i + 1) % n][0], ring[(i + 1) % n][1]
        if (y1 > y) != (y2 > y) and x < (x2 - x1) * (y - y1) / (y2 - y1) + x1:
            ins = not ins
    return ins


def hranica_mesta():
    """Prstence administratívnej hranice mesta zo súboru vedľa notebooku."""
    p = Path(HRANICA_MESTA)
    if not p.exists():
        print(f"  ! {p} chýba — filter 'iba v meste' sa preskočí")
        return []
    gj = json.loads(p.read_text(encoding="utf-8"))
    ringy = []
    for f in gj.get("features", []):
        g = f.get("geometry") or {}
        if g.get("type") == "Polygon":
            ringy.append(g["coordinates"][0])
        elif g.get("type") == "MultiPolygon":
            ringy += [part[0] for part in g["coordinates"]]
    return ringy


def v_meste(coords, ringy):
    """Leží stred hrany vnútri hranice mesta?"""
    if not ringy:
        return True
    if not coords:
        return False
    x, y = coords[len(coords) // 2][:2]
    return any(_v_polygone(x, y, r) for r in ringy)


def brany_s_objemom(linky, radius_m=1500.0):
    """Brány (externé centroidy) a objem na koridore, ktorý cez ne vedie.

    Konektory zón API nevydáva, preto sa berie najvyťaženejšia hrana do `radius_m`
    od brány, ktorá má rovnaký `osm_ref` ako brána (`matched_ref`).
    """
    import math

    gj = get_json("/api/centroids")
    if not gj:
        return pd.DataFrame()

    stredy = []
    for _, r in linky.iterrows():
        c = r.get("_coords")
        if not c:
            continue
        x, y = c[len(c) // 2][0], c[len(c) // 2][1]
        stredy.append((x, y, r.get("osm_ref"), r.get("wd_daily_tot"), r.get("link_type")))

    def refy(v):
        return {str(z).replace("/", "").replace(" ", "").upper()
                for z in str(v or "").split(";") if str(z).strip()}

    riadky = []
    for f in gj.get("features", []):
        p = f.get("properties") or {}
        if not p.get("is_external"):
            continue
        lon, lat = f["geometry"]["coordinates"]
        kx = 111320.0 * math.cos(math.radians(lat))
        hladane = refy(p.get("matched_ref"))
        objem, typ = 0.0, None
        for x, y, ref, vol, lt in stredy:
            if math.hypot((x - lon) * kx, (y - lat) * 110540.0) > radius_m:
                continue
            if hladane and not (hladane & refy(ref)):
                continue
            v = float(pd.to_numeric(vol, errors="coerce") or 0)
            if v > objem:
                objem, typ = v, lt
        riadky.append({"brana": p.get("gateway_name"), "ref": p.get("matched_ref"),
                       "lat": lat, "lon": lon, "objem": objem, "link_type": typ})
    return pd.DataFrame(riadky).sort_values("objem", ascending=False)


def uloz_json(obj, meno):
    p = OUT / meno
    p.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"  → {p.name}")


meta = get_json("/api/meta")
if meta is None:
    raise SystemExit("API nebeží — spusti kontajner a pusti bunku znova.")
print("✅", meta.get("place_name"), "| center", meta.get("map_center"))

✅ Brno, Czechia | center {'lat': 49.201834914660175, 'lng': 16.58811554801459}


---
## 1. Reporty z pipeline

Surové JSON sa uložia bez úprav, aby bola k dispozícii úplná stopa.

In [3]:
validacia  = get_json("/api/reports/validation")
kalibracia = get_json("/api/reports/calibration")
od_sumar   = get_json("/api/reports/od-summary")
siet_sumar = get_json("/api/reports/network")      # môže chýbať, netreba

for obj, meno in [(validacia, "validation_report.json"),
                  (kalibracia, "calibration_report.json"),
                  (od_sumar, "od_summary.json"),
                  (siet_sumar, "network_summary.json")]:
    if obj is not None:
        uloz_json(obj, meno)

  → validation_report.json
  → calibration_report.json
  → od_summary.json
  → network_summary.json


---
## 2. Sieť a zóny — porovnanie s DP

In [4]:
# všetky linky (bez filtra) — potrebné pre VHT/VKT aj pre počty
linky = geojson_to_df(get_json("/api/links", min_volume=0))
uzly  = geojson_to_df(get_json("/api/nodes"))
zony  = geojson_to_df(get_json("/api/zones"))

M["siet"] = {
    "hrany": int(len(linky)),
    "uzly": int(len(uzly)),
    "zony_taz": int(len(zony)),
}
if "is_external" in zony.columns:
    ext = pd.to_numeric(zony["is_external"], errors="coerce").fillna(0).astype(int)
    M["siet"]["zony_externe"] = int(ext.sum())
    M["siet"]["zony_interne"] = int((ext == 0).sum())

DP_SIET = {"hrany": 36105, "uzly": 32539, "zony_taz": 92,
           "zony_interne": 87, "zony_externe": 5}

porov_siet = pd.DataFrame([
    {"veličina": k, "model": M["siet"].get(k), "DP": v,
     "rozdiel": (M["siet"].get(k) - v) if M["siet"].get(k) is not None else None}
    for k, v in DP_SIET.items()
])
print("SIEŤ A ZÓNY")
display(porov_siet)

  ✗ /api/nodes: 500 Internal Server Error
SIEŤ A ZÓNY


,veličina,model,DP,rozdiel
0,hrany,36435,36105,330
1,uzly,0,32539,-32539
2,zony_taz,69,92,-23
3,zony_interne,64,87,-23
4,zony_externe,5,5,0


---
## 3. Dopyt — OD matica po segmentoch

In [5]:
if od_sumar:
    seg = od_sumar.get("segments", {}) or {}
    M["dopyt"] = {
        "od_spolu":            seg.get("combined_daily"),
        "commuting":           seg.get("commuting"),
        "other":               seg.get("other"),
        "external_local":      seg.get("external_local"),
        "external_through":    seg.get("external_through_total"),
        "external_spolu":      seg.get("external_total"),
    }

    DP_DOPYT = {"od_spolu": 641622, "other": 94147,
                "external_local": 130000, "external_through": 36610}

    porov_dopyt = pd.DataFrame([
        {"segment": k, "model": M["dopyt"].get(k), "DP": DP_DOPYT.get(k),
         "rozdiel_%": (round((M["dopyt"][k] - DP_DOPYT[k]) / DP_DOPYT[k] * 100, 1)
                       if k in DP_DOPYT and M["dopyt"].get(k) else None)}
        for k in M["dopyt"]
    ])
    print("DOPYT [voz./deň]")
    display(porov_dopyt)
else:
    print("od_summary.json nedostupný")

DOPYT [voz./deň]


,segment,model,DP,rozdiel_%
0,od_spolu,556040.0,641622.0,-13.3
1,commuting,297113.0,NaN,NaN
2,other,93035.0,94147.0,-1.2
3,external_local,130000.0,130000.0,0.0
4,external_through,35891.0,36610.0,-2.0
5,external_spolu,165891.0,NaN,NaN


---
## 4. Kalibrácia a validácia

**Pozor na dve sady čísel.** `calibration_reference` sú dáta, ktoré ODME videla
(65 % CSD úsekov), `holdout_validation` sú tie, ktoré nevidela (35 %). Verdikt sa
berie z holdoutu. DP zrejme uvádza kalibračnú časť — porovnávaj správny stĺpec.

In [6]:
if validacia:
    b  = validacia.get("benchmarks", {}) or {}
    cf = b.get("calibration_fit", {}) or {}
    hv = b.get("holdout_validation", {}) or {}
    cr = validacia.get("calibration_reference", {}) or {}

    M["kalibracia"] = {
        "n":          cr.get("n"),
        "matched":    cr.get("matched"),
        "r2":         cf.get("r2"),
        "slope":      cf.get("slope"),
        "pct_rmse":   cf.get("pct_rmse"),
        "bias_pct":   cf.get("bias_abs_pct"),
    }
    M["holdout"] = {
        "r2":         hv.get("r2"),
        "slope":      hv.get("slope"),
        "pct_rmse":   hv.get("pct_rmse"),
        "bias_pct":   hv.get("bias_abs_pct"),
        "adekvatnost": b.get("holdout_adequacy"),
        "prijate":     hv.get("overall_pass"),
    }
    M["screenlines"] = {
        "max_odchylka_pct": b.get("screenline_max_error_pct"),
        "porovnanych":      b.get("screenline_n_compared"),
        "vylucenych":       b.get("screenline_n_excluded"),
    }
    M["verdikt"] = {
        "zdroj":       b.get("verdict_source"),
        "overall_pass": b.get("overall_pass"),
        "quality_gates": validacia.get("quality_gates"),
    }

    DP_FIT = {"r2": 0.83, "slope": 0.94, "pct_rmse": 40.4, "bias_pct": 8.6}
    CIEL   = {"r2": "≥ 0.80", "slope": "0.85–1.15",
              "pct_rmse": "≤ 35", "bias_pct": "≤ 15"}

    tab = pd.DataFrame([
        {"metrika": k,
         "kalibračná": M["kalibracia"].get(k),
         "holdout": M["holdout"].get(k),
         "DP §4": DP_FIT.get(k),
         "cieľ": CIEL.get(k)}
        for k in ("r2", "slope", "pct_rmse", "bias_pct")
    ])
    print("ZHODA S POZOROVANÍM")
    display(tab)
    print(f"n kalibračných bodov : {M['kalibracia']['n']}  "
          f"(adekvátnosť holdoutu: {M['holdout']['adekvatnost']})")
    print(f"screenlines          : max odchýlka {M['screenlines']['max_odchylka_pct']} %, "
          f"{M['screenlines']['porovnanych']} porovnaných")
    print(f"verdikt              : {M['verdikt']['overall_pass']} "
          f"(zdroj: {M['verdikt']['zdroj']})")

ZHODA S POZOROVANÍM


,metrika,kalibračná,holdout,DP §4,cieľ
0,r2,0.9028,0.6270,0.83,≥ 0.80
1,slope,1.0026,0.7443,0.94,0.85–1.15
2,pct_rmse,30.2000,47.0000,40.40,≤ 35
3,bias_pct,1.5100,11.8000,8.60,≤ 15


n kalibračných bodov : 14  (adekvátnosť holdoutu: thin)
screenlines          : max odchýlka 88.6 %, 21 porovnaných
verdikt              : False (zdroj: holdout)


In [7]:
# ODME — koľko musela pokriviť maticu (3. úroveň porovnania z PLAN.md)
if kalibracia:
    sd = kalibracia.get("seed_deviation", {}) or {}
    M["odme"] = {
        "iteracii":            kalibracia.get("iterations"),
        "konvergovalo":        kalibracia.get("converged"),
        "metoda":              kalibracia.get("method"),
        "drift_od_seedu_pct":  sd.get("prior_drift_pct"),
        "buniek_nad_prahom":   sd.get("n_cells_exceeding_threshold"),
        "max_pomer_bunky":     sd.get("max_cell_ratio"),
        "seed_spolu":          sd.get("total_seed"),
        "final_spolu":         sd.get("total_final"),
    }
    hist = kalibracia.get("history", []) or []
    if hist:
        M["odme"]["Z_start"] = hist[0].get("Z_objective")
        M["odme"]["Z_koniec"] = min(h.get("Z_objective", float("inf")) for h in hist)
    print("ODME")
    display(pd.Series(M["odme"]).to_frame("hodnota"))
    print("\nČím menší |drift_od_seedu_pct| a menej buniek nad prahom, tým lepší bol vstupný odhad.")

ODME


,hodnota
iteracii,22
konvergovalo,False
metoda,entropy_odme
drift_od_seedu_pct,None
buniek_nad_prahom,None
max_pomer_bunky,None
seed_spolu,None
final_spolu,None
Z_start,26409580.5
Z_koniec,2454995.2



Čím menší |drift_od_seedu_pct| a menej buniek nad prahom, tým lepší bol vstupný odhad.


---
## 5. Baseline sieťové KPI

In [8]:
def vht(df, vol="wd_daily_tot", t="Congested_Time_Max"):
    if vol not in df.columns or t not in df.columns:
        return None
    v = pd.to_numeric(df[vol], errors="coerce").fillna(0)
    tt = pd.to_numeric(df[t], errors="coerce").fillna(0)
    return float((v * tt / 3600.0).sum())


def vkt(df, vol="wd_daily_tot"):
    if vol not in df.columns or "distance" not in df.columns:
        return None
    v = pd.to_numeric(df[vol], errors="coerce").fillna(0)
    d = pd.to_numeric(df["distance"], errors="coerce").fillna(0) / 1000.0
    return float((v * d).sum())


voc = pd.to_numeric(linky.get("VOC_max"), errors="coerce").fillna(0)
M["baseline_kpi"] = {
    "vht_voz_h":        round(vht(linky) or 0, 1),
    "vkt_voz_km":       round(vkt(linky) or 0, 1),
    "pretazenych_hran": int((voc > 1.0).sum()),
    "objem_spolu":      round(float(pd.to_numeric(linky.get("wd_daily_tot"),
                                                  errors="coerce").fillna(0).sum()), 0),
}
if "LOS_max" in linky.columns:
    M["baseline_kpi"]["los"] = linky["LOS_max"].value_counts().to_dict()

display(pd.Series({k: v for k, v in M["baseline_kpi"].items()
                   if not isinstance(v, dict)}).to_frame("hodnota"))
if "los" in M["baseline_kpi"]:
    print("\nRozdelenie LOS:", M["baseline_kpi"]["los"])

# uložiť objemy na linkoch (bez geometrie — tá je veľká)
linky.drop(columns=["_coords"]).to_parquet(OUT / "links_baseline.parquet", index=False)
print(f"\n  → links_baseline.parquet ({len(linky):,} hrán)")

,hodnota
vht_voz_h,9595749.7
vkt_voz_km,6266210.1
pretazenych_hran,259.0
objem_spolu,62716276.0



Rozdelenie LOS: {'A': 32736, 'B': 1887, 'C': 946, 'D': 398, 'F': 259, 'E': 209}

  → links_baseline.parquet (36,435 hrán)


---
## 6. Scenáre uzávierky

Púšťajú sa dva:

- **celá ulica** (`SCENAR_ULICA`) — hlavný výsledok
- **null scenár** (`SCENAR_NULL_LINK`) — hrana s nulovým objemom; jej „uzávierka" nemôže
  nič presunúť, takže výsledná redistribúcia je **odhad šumu** z opakovaného priraďovania.
  Od skutočného efektu sa má odpočítať.

> `link_id` nie sú stabilné medzi prestavbami siete — scenár definuj názvom ulice, nie ID.


In [ ]:
def spusti_scenar(link_ids, lanes_map=None, typ="full", zostane=0):
    """Spustí scenár a počká naň. Vráti job id."""
    lanes_map = lanes_map or {}
    payload = {"links": [{
        "link_id": int(l), "direction": "both", "closure_type": typ,
        "lanes": max(int(lanes_map.get(int(l), 2)), 1),
        "lanes_remaining": int(zostane),
    } for l in link_ids]}
    r = requests.post(f"{API_URL}/api/scenarios/run", json=payload, timeout=60)
    if r.status_code not in (200, 202):
        raise RuntimeError(f"{r.status_code}: {r.text[:400]}")
    jid = r.json()["id"]
    t0 = time.time()
    while time.time() - t0 < 3600:
        st = requests.get(f"{API_URL}/api/scenarios/{jid}/status", timeout=15).json()
        print(f"\r  {st['status']} ({st['elapsed_seconds']:.0f}s)   ", end="")
        if st["status"] == "done":
            print(); return jid
        if st["status"] == "error":
            raise RuntimeError(st.get("error"))
        time.sleep(3)
    raise TimeoutError


def num(df, col):
    return pd.to_numeric(df.get(col), errors="coerce").fillna(0)


def kpi_scenara(res, zavrete):
    dv = num(res, "delta_vol")
    bv, sv = num(res, "baseline_vol"), num(res, "wd_daily_tot")
    ct = num(res, "Congested_Time_Max") / 3600.0
    zav = res["link_id"].isin(zavrete)
    return {
        "zavretych_hran": len(zavrete),
        "objem_na_zavretych_pred": round(float(bv[zav].sum())),
        "objem_na_zavretych_po": round(float(sv[zav].sum())),
        "hran_nad_500": int((dv.abs() > 500).sum()),
        "hran_nad_100": int((dv.abs() > 100).sum()),
        "redistribucia": round(float(dv.abs().sum())),
        "max_narast": round(float(dv.max())),
        "max_pokles": round(float(dv.min())),
        "vht_scenar": round(float((sv * ct).sum()), 1),
        "vkt_scenar": round(float((sv * num(res, "distance") / 1000).sum()), 1),
    }


SCENARE = {}

# --- null scenár: odhad šumu ---
try:
    jid = spusti_scenar([SCENAR_NULL_LINK])
    res = geojson_to_df(get_json(f"/api/scenarios/{jid}/results"))
    SCENARE["null"] = {"res": res, "zavrete": [SCENAR_NULL_LINK],
                       "kpi": kpi_scenara(res, [SCENAR_NULL_LINK])}
    k = SCENARE["null"]["kpi"]
    print(f"null scenár (link {SCENAR_NULL_LINK}): objem pred = {k['objem_na_zavretych_pred']:,.0f}")
    print(f"  → šum: redistribúcia {k['redistribucia']:,.0f}, {k['hran_nad_500']:,} hrán > 500")
except Exception as e:
    print("null scenár zlyhal:", e)


In [ ]:
# --- hlavný scenár: celá ulica ---
ulica = linky[linky["name"].astype(str).str.contains(SCENAR_ULICA, na=False)]
if "link_type" in ulica.columns and (ulica["link_type"] == "secondary").any():
    ulica = ulica[ulica["link_type"] == "secondary"]
# Jihlavská pokračuje za hranicou Brna ako II/602 do extravilánu. Bez tohto filtra
# scenár zatvára aj ten úsek, čo je iná (a oveľa väčšia) intervencia než uzávierka
# mestskej ulice — a vyrába umelé objazdy cez celé mesto.
if IBA_V_MESTE:
    ringy = hranica_mesta()
    pred = len(ulica)
    ulica = ulica[ulica["_coords"].map(lambda c: v_meste(c, ringy))]
    print(f"  filter 'iba v meste': {pred} → {len(ulica)} hrán")

ids = ulica["link_id"].astype(int).tolist()
lanes_map = (dict(zip(ulica["link_id"].astype(int), num(ulica, "lanes").replace(0, 2).astype(int)))
             if "lanes" in ulica.columns else {})

print(f"{SCENAR_ULICA}: {len(ids)} hrán, "
      f"{num(ulica, 'distance').sum():,.0f} m, "
      f"objem {num(ulica, 'wd_daily_tot').sum():,.0f} voz/deň")

if ids:
    try:
        jid = spusti_scenar(ids, lanes_map=lanes_map)
        res = geojson_to_df(get_json(f"/api/scenarios/{jid}/results"))
        SCENARE[SCENAR_ULICA] = {"res": res, "zavrete": ids, "kpi": kpi_scenara(res, ids)}
        print(f"scenár {SCENAR_ULICA}: hotovo")
    except Exception as e:
        print(f"scenár {SCENAR_ULICA} zlyhal: {e}")
else:
    print("Ulica sa v sieti nenašla — skontroluj SCENAR_ULICA.")


In [ ]:
M["scenare"] = {k: v["kpi"] for k, v in SCENARE.items()}

for meno, s in SCENARE.items():
    bez = meno.replace(" ", "_").replace("/", "_")
    s["res"].drop(columns=["_coords"], errors="ignore").to_parquet(
        OUT / f"scenario_{bez}.parquet", index=False)
    print(f"  → scenario_{bez}.parquet")

if M["scenare"]:
    t = pd.DataFrame(M["scenare"]).T
    display(t)

    # efekt očistený o šum z null scenára
    if "null" in M["scenare"] and len(M["scenare"]) > 1:
        sum_ = M["scenare"]["null"]
        print("\nEfekt očistený o šum (null scenár):")
        for meno, k in M["scenare"].items():
            if meno == "null":
                continue
            print(f"  {meno}: redistribúcia {k['redistribucia']:,.0f} "
                  f"− šum {sum_['redistribucia']:,.0f} "
                  f"= {k['redistribucia'] - sum_['redistribucia']:,.0f} voz/deň")
            print(f"    hrán > 500: {k['hran_nad_500']:,} − {sum_['hran_nad_500']:,} "
                  f"= {k['hran_nad_500'] - sum_['hran_nad_500']:,}")


---
## 7. Mapy

In [ ]:
# ═══ GENERÁTOR MÁP ════════════════════════════════════════════════════
# Vlastná Leaflet stránka namiesto folium: dáta sú v JSON raz a prekresľuje ich JS,
# takže sa dá prepínať deň v týždni aj základný stav bez duplikovania geometrie.

DNI_PORADIE = ["priemer", "pondelok", "utorok", "streda", "štvrtok", "piatok", "sobota", "nedeľa"]


def faktory_dni():
    """(lokálne, externé) denné faktory.

    Lokálna doprava sa škáluje faktormi, ktoré sa model naučil z CSD
    (`/api/temporal/profile`, rozlíšenie len pracovný deň / sobota / nedeľa),
    externá a tranzitná faktormi nerezidentov z kamdojizdime (7 dní).
    Oboje normalizované na pracovný deň = 1.0, takže „priemer“ = dnešný výstup modelu.
    """
    prof = get_json("/api/temporal/profile") or {}
    df = prof.get("day_factors", {}) or {}
    prac = float(df.get("workday", 1.0)) or 1.0
    lok = {d: 1.0 for d in DNI_PORADIE}
    lok["sobota"] = round(float(df.get("saturday", prac)) / prac, 4)
    lok["nedeľa"] = round(float(df.get("sunday", prac)) / prac, 4)

    ext = {d: 1.0 for d in DNI_PORADIE}
    p = Path("kamdojizdime_dni.json")
    if p.exists():
        ext.update(json.loads(p.read_text(encoding="utf-8"))["faktory"])
    else:
        print("  ! kamdojizdime_dni.json chýba — externý segment sa nebude škálovať")
    ext["priemer"] = 1.0
    return lok, ext


def tranzitne_podiely():
    """link_id → podiel tranzitu na objeme (z /api/diagnostics/through-traffic)."""
    gj = get_json("/api/diagnostics/through-traffic")
    out = {}
    for f in ((gj or {}).get("links") or {}).get("features", []):
        p = f.get("properties") or {}
        try:
            out[int(p["link_id"])] = max(0.0, min(1.0, float(p.get("through_share") or 0)))
        except Exception:
            continue
    return out


def _hrany_pre_mapu(df, podiely, min_objem, min_delta=None):
    """Zbalí hrany do kompaktných záznamov pre JS."""
    von = []
    for _, r in df.iterrows():
        c = r.get("_coords")
        if not c:
            continue
        base = float(pd.to_numeric(r.get("baseline_vol"), errors="coerce") or 0)
        scen = float(pd.to_numeric(r.get("wd_daily_tot"), errors="coerce") or 0)
        if min_delta is not None and abs(scen - base) < min_delta and max(base, scen) < min_objem:
            continue
        if min_delta is None and scen < min_objem:
            continue
        try:
            lid = int(r.get("link_id"))
        except Exception:
            lid = -1
        von.append({
            "id": lid,
            "n": popis_hrany(r),
            "t": str(r.get("link_type") or ""),
            "b": round(base, 1),
            "s": round(scen, 1),
            "ts": round(podiely.get(lid, 0.0), 3),
            "v": round(float(pd.to_numeric(r.get("VOC_max"), errors="coerce") or 0), 3),
            "l": str(r.get("LOS_max") or "").strip(),
            "c": [[round(x, 5), round(y, 5)] for x, y in c],
        })
    return von


_HTML = r"""<!doctype html><html lang="sk"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1"><title>__TITUL__</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
 :root{--bg:#fff;--fg:#16181d;--mut:#6b7280;--line:#e3e6ea;--panel:rgba(255,255,255,.97)}
 *{box-sizing:border-box} html,body{margin:0;height:100%;background:var(--bg);color:var(--fg);
  font:14px/1.45 -apple-system,BlinkMacSystemFont,"Segoe UI",system-ui,sans-serif}
 #map{position:absolute;inset:0}
 .box{position:absolute;z-index:1000;background:var(--panel);border:1px solid var(--line);
  border-radius:12px;box-shadow:0 6px 24px rgba(0,0,0,.16);backdrop-filter:blur(8px)}
 #ovl{top:14px;left:14px;padding:14px 16px;max-width:310px}
 #leg{bottom:14px;left:14px;padding:12px 14px;min-width:210px}
 h1{margin:0 0 2px;font-size:15px;font-weight:650;letter-spacing:-.01em}
 .sub{color:var(--mut);font-size:12px;margin-bottom:12px}
 label.f{display:block;font-size:11px;font-weight:600;text-transform:uppercase;
  letter-spacing:.06em;color:var(--mut);margin:12px 0 5px}
 select{width:100%;padding:7px 9px;border:1px solid var(--line);border-radius:8px;
  background:var(--bg);color:var(--fg);font:inherit;font-size:13px}
 .seg{display:flex;gap:3px;background:var(--line);padding:3px;border-radius:9px}
 .seg button{flex:1;padding:6px 4px;border:0;border-radius:7px;background:transparent;
  color:var(--mut);font:inherit;font-size:12px;font-weight:550;cursor:pointer}
 .seg button[aria-pressed=true]{background:var(--bg);color:var(--fg);box-shadow:0 1px 3px rgba(0,0,0,.13)}
 .chk{display:flex;align-items:center;gap:8px;margin:7px 0;font-size:13px;cursor:pointer}
 .chk input{accent-color:#4b7bec;width:15px;height:15px}
 .row{display:flex;justify-content:space-between;gap:12px;font-size:12px;padding:2px 0}
 .row b{font-variant-numeric:tabular-nums}
 .sc{height:9px;border-radius:5px;margin:6px 0 4px}
 .ends{display:flex;justify-content:space-between;color:var(--mut);font-size:11px}
 .swatch{display:inline-block;width:22px;height:4px;border-radius:2px;vertical-align:middle;margin-right:7px}
 .note{color:var(--mut);font-size:11px;margin-top:9px;line-height:1.4}
 .leaflet-tooltip{font:12px/1.4 system-ui;border-radius:8px;border:1px solid var(--line);
  background:var(--panel);color:var(--fg);box-shadow:0 3px 12px rgba(0,0,0,.2)}
</style></head><body>
<div id="map"></div>
<div class="box" id="ovl">
 <h1>__TITUL__</h1><div class="sub">__PODTITUL__</div>
 <div class="seg" id="mode">__MODE_BUTTONS__</div>
 <label class="f" for="den">Deň v týždni</label>
 <select id="den">__DAY_OPTIONS__</select>
 <div id="dayinfo" class="note"></div>
 <label class="f">Vrstvy</label>
 <label class="chk"><input type="checkbox" id="cbClosed" checked> uzavreté hrany</label>
 <label class="chk"><input type="checkbox" id="cbGw" checked> brány (vstupy do modelu)</label>
</div>
<div class="box" id="leg"><div id="legbody"></div></div>
<script>
const D = __DATA__;
const map = L.map('map',{preferCanvas:true,zoomControl:true}).setView(D.stred, 12);
L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',
 {maxZoom:19, attribution:'© OpenStreetMap, © CARTO'}).addTo(map);

const gLinks=L.layerGroup().addTo(map), gClosed=L.layerGroup().addTo(map), gGw=L.layerGroup().addTo(map);
let mode = D.mody[0], den = 'priemer';

const nf = n => Math.round(n).toLocaleString('sk-SK').replace(/ /g,' ');
const fac = l => { const fl=D.flok[den]??1, fe=D.fext[den]??1; return (1-l.ts)*fl + l.ts*fe; };
const vol = (l,ktore) => (ktore==='b'? l.b : l.s) * fac(l);

const RAMP=['#dbeafe','#93c5fd','#60a5fa','#3b82f6','#2563eb','#1d4ed8','#1e3a8a'];
const LOS={A:'#2c7fb8',B:'#41b6c4',C:'#a1dab4',D:'#fecc5c',E:'#fd8d3c',F:'#e31a1c'};
const losOf = v => v<0.6?'A':v<0.7?'B':v<0.8?'C':v<0.9?'D':v<1?'E':'F';
function farbaObjem(v,max){ if(v<=0) return '#c8ccd2';
 const t=Math.min(1,Math.log10(1+v)/Math.log10(1+Math.max(max,10)));
 return RAMP[Math.min(RAMP.length-1,Math.floor(t*RAMP.length))]; }
function farbaDelta(d){ return d>0?'#d7191c':'#2c7bb6'; }
const PRAZDNE='#7b3294';   // doprava klesla prakticky na nulu — nie je to len pokles
const ZAVRETE=new Set(D.zavrete_ids||[]);
const vyprazdnena = l => !ZAVRETE.has(l.id) && vol(l,'b')>=100 && vol(l,'s')<1;
const sirka=(v,max,lo,hi)=>lo+(hi-lo)*Math.min(1,Math.abs(v)/(max||1));

function kresli(){
 gLinks.clearLayers();
 const dv = D.hrany.map(l=>vol(l,'s')-vol(l,'b'));
 const maxD = kvantil(dv.map(Math.abs),0.95)||1;
 const maxV = kvantil(D.hrany.map(l=>Math.max(vol(l,'s'),vol(l,'b'))),0.98)||1;
 D.hrany.forEach((l,i)=>{
  let farba,w,d=dv[i],hodnota;
  if(mode==='delta'){ if(Math.abs(d)<100) return;
        const prazdna=vyprazdnena(l);
        farba = prazdna ? PRAZDNE : farbaDelta(d);
        w = prazdna ? Math.max(sirka(d,maxD,1.4,9), 4) : sirka(d,maxD,1.4,9); hodnota=d; }
  else { const v=vol(l, mode==='base'?'b':'s'); if(v<D.min_objem) return;
         farba = mode==='los' ? (LOS[l.l]||LOS[losOf(l.v)]||'#999') : farbaObjem(v,maxV);
         w=sirka(v,maxV,0.9,8); hodnota=v; }
  L.polyline(l.c.map(p=>[p[1],p[0]]),{color:farba,weight:w,opacity:.85,lineCap:'round'})
   .bindTooltip(`<b>${l.n}</b>${mode==='delta'&&vyprazdnena(l)?' <span style="color:'+PRAZDNE+'">· vyprázdnená</span>':''}`
     + `<br>${nf(vol(l,'b'))} → ${nf(vol(l,'s'))} voz/deň`
     + (Math.abs(d)>=1?`<br><b>${d>0?'+':''}${nf(d)}</b>`:'')
     + (l.ts>0?`<br><span style="opacity:.7">tranzit ${Math.round(l.ts*100)} %</span>`:''),{sticky:true})
   .addTo(gLinks);
 });
 legenda(maxD,maxV);
 const fl=D.flok[den]??1, fe=D.fext[den]??1;
 document.getElementById('dayinfo').textContent = den==='priemer'
  ? 'Priemerný pracovný deň — priamy výstup modelu.'
  : `Objemy × ${fl.toFixed(2)} (lokálna doprava, CSD) a × ${fe.toFixed(2)} (externá a tranzit, kamdojizdime).`;
}
function kvantil(a,q){ const s=a.filter(x=>isFinite(x)).sort((x,y)=>x-y); if(!s.length) return 0;
 return s[Math.min(s.length-1,Math.floor(q*s.length))]; }

function legenda(maxD,maxV){
 const b=document.getElementById('legbody');
 if(mode==='delta'){
  b.innerHTML = `<div class="row"><span><i class="swatch" style="background:#d7191c"></i>nárast</span></div>
   <div class="row"><span><i class="swatch" style="background:#2c7bb6"></i>pokles</span></div>
   <div class="row"><span><i class="swatch" style="background:${PRAZDNE}"></i>vyprázdnené (na 0)</span></div>
   <div class="row"><span><i class="swatch" style="background:#111"></i>uzavreté</span></div>
   <div class="note">Hrúbka = veľkosť zmeny (strop ${nf(maxD)} voz/deň).<br>Zobrazené |Δ| ≥ 100 voz/deň.</div>`;
 } else if(mode==='los'){
  b.innerHTML = Object.entries(LOS).map(([k,c])=>
   `<div class="row"><span><i class="swatch" style="background:${c}"></i>LOS ${k}</span></div>`).join('')
   + `<div class="note">Hrúbka = objem.</div>`;
 } else {
  b.innerHTML = `<div class="sub" style="margin:0 0 4px">${mode==='base'?'Objem pred uzávierkou':'Objem po uzávierke'}</div>
   <div class="sc" style="background:linear-gradient(90deg,${RAMP.join(',')})"></div>
   <div class="ends"><span>${nf(D.min_objem)}</span><span>${nf(maxV)}+ voz/deň</span></div>
   <div class="note">Logaritmická škála.</div>`;
 }
}

D.zavrete.forEach(l=>L.polyline(l.c.map(p=>[p[1],p[0]]),
 {color:'#111',weight:9,opacity:.95}).bindTooltip(`uzavreté — ${l.n}`,{sticky:true}).addTo(gClosed));

D.brany.forEach(b=>{
 L.circleMarker([b.lat,b.lon],{radius:9,color:'#111',weight:2,fillColor:'#ffd24d',fillOpacity:.95})
  .bindTooltip(`<b>brána ${b.brana}</b><br>${b.ref} · ${b.link_type}<br>${nf(b.objem)} voz/deň na koridore`,
   {sticky:true}).addTo(gGw);
 L.marker([b.lat,b.lon],{interactive:false,icon:L.divIcon({className:'',html:
  `<div style="font:650 11px/1.15 system-ui;color:var(--fg);text-shadow:0 0 3px var(--bg),0 0 3px var(--bg);
   transform:translate(13px,-7px);white-space:nowrap">${b.brana}<br>
   <span style="font-weight:400;opacity:.75">${nf(b.objem)}</span></div>`})}).addTo(gGw);
});

document.querySelectorAll('#mode button').forEach(btn=>btn.onclick=()=>{
 mode=btn.dataset.m;
 document.querySelectorAll('#mode button').forEach(x=>x.setAttribute('aria-pressed', x===btn));
 kresli();
});
document.getElementById('den').onchange = e => { den=e.target.value; kresli(); };
document.getElementById('cbClosed').onchange = e => e.target.checked?gClosed.addTo(map):map.removeLayer(gClosed);
document.getElementById('cbGw').onchange = e => e.target.checked?gGw.addTo(map):map.removeLayer(gGw);
kresli();
</script></body></html>"""


def uloz_mapu(cesta, titul, podtitul, hrany, brany, zavrete, mody, min_objem, popisy=None):
    """Zapíše samostatnú Leaflet stránku (bez folium)."""
    lok, ext = FAKTORY_DNI
    popisy = {"delta": "zmena", "scen": "objem po", "base": "objem pred", "los": "LOS", **(popisy or {})}
    stred = [49.195, 16.606]
    if hrany:
        xs = [p[0] for h in hrany[:400] for p in h["c"]]
        ys = [p[1] for h in hrany[:400] for p in h["c"]]
        stred = [sum(ys) / len(ys), sum(xs) / len(xs)]
    data = {"hrany": hrany, "brany": brany, "zavrete": zavrete, "stred": stred,
            "zavrete_ids": [z["id"] for z in zavrete if z.get("id") is not None],
            "flok": lok, "fext": ext, "mody": mody, "min_objem": min_objem}
    html = (_HTML
            .replace("__TITUL__", titul)
            .replace("__PODTITUL__", podtitul)
            .replace("__MODE_BUTTONS__", "".join(
                f'<button data-m="{m}" aria-pressed="{str(i == 0).lower()}">{popisy[m]}</button>'
                for i, m in enumerate(mody)))
            .replace("__DAY_OPTIONS__", "".join(
                f'<option value="{d}">{d}</option>' for d in DNI_PORADIE))
            .replace("__DATA__", json.dumps(data, ensure_ascii=False, separators=(",", ":"))))
    Path(cesta).write_text(html, encoding="utf-8")
    print(f"  → {Path(cesta).name}  ({len(html)//1024:,} kB, {len(hrany):,} hrán)".replace(",", " "))


FAKTORY_DNI = faktory_dni()
PODIELY_TRANZIT = tranzitne_podiely()
print(f"denné faktory: lokálne {FAKTORY_DNI[0]['sobota']}/{FAKTORY_DNI[0]['nedeľa']} (so/ne), "
      f"externé {FAKTORY_DNI[1]['sobota']}/{FAKTORY_DNI[1]['nedeľa']}; "
      f"tranzitný podiel známy pre {len(PODIELY_TRANZIT):,} hrán".replace(",", " "))


In [ ]:
# --- baseline mapa ---
linky["baseline_vol"] = num(linky, "wd_daily_tot")
hrany = _hrany_pre_mapu(linky, PODIELY_TRANZIT, MAPA_MIN_OBJEM)
BRANY = brany_s_objemom(linky)
uloz_mapu(OUT / "mapa_baseline.html",
          f"{meta.get('title_short', 'Model')} — základný stav",
          f"Denné objemy z priradenia. Zobrazené hrany ≥ {MAPA_MIN_OBJEM:,} voz/deň.".replace(",", " "),
          hrany, BRANY.to_dict("records"), [], ["scen", "los"], MAPA_MIN_OBJEM,
          popisy={"scen": "objem"})
if not BRANY.empty:
    display(BRANY[["brana", "ref", "link_type", "objem"]].reset_index(drop=True))


In [ ]:
# --- mapy scenárov ---
if SCENARE:
    for meno, s in SCENARE.items():
        res = s["res"]
        hrany = _hrany_pre_mapu(res, PODIELY_TRANZIT, MAPA_MIN_OBJEM, min_delta=100)
        zavrete = [{"id": int(r["link_id"]), "n": popis_hrany(r),
                    "c": [[round(x, 5), round(y, 5)] for x, y in r["_coords"]]}
                   for _, r in res[res["link_id"].isin(s["zavrete"])].iterrows() if r["_coords"]]
        bez = meno.replace(" ", "_").replace("/", "_")
        uloz_mapu(OUT / f"mapa_scenar_{bez}.html",
                  f"Uzávierka {meno}",
                  f"{len(s['zavrete'])} uzavretých hrán · prepni režim, deň alebo vrstvy vľavo hore.",
                  hrany, BRANY.to_dict("records"), zavrete,
                  ["delta", "scen", "base"], MAPA_MIN_OBJEM)
else:
    print("Žiadne scenáre na zobrazenie.")


---
## 8. Uloženie

In [14]:
def git(*args):
    try:
        return subprocess.check_output(["git", *args], cwd="..",
                                       stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return None


cfg_txt = Path("../config/brno/sim.yaml").read_text(encoding="utf-8")

M["_meta"] = {
    "variant": VARIANT,
    "popis": POPIS,
    "vytvorene": datetime.now().isoformat(timespec="seconds"),
    "api_url": API_URL,
    "mesto": meta.get("place_name"),
    "git_commit": git("rev-parse", "--short", "HEAD"),
    "git_dirty": bool(git("status", "--porcelain")),
    "config_sha256": hashlib.sha256(cfg_txt.encode()).hexdigest()[:16],
}

uloz_json(M, "metrics.json")
(OUT / "sim.yaml.snapshot").write_text(cfg_txt, encoding="utf-8")
print("  → sim.yaml.snapshot")

print(f"\n✅ Uložené do {OUT.resolve()}\n")
for p in sorted(OUT.iterdir()):
    print(f"   {p.name:42s} {p.stat().st_size/1024:>9,.0f} kB")

  → metrics.json
  → sim.yaml.snapshot

✅ Uložené do /Users/magdalena.ondruskova/git/personal/Traffic-sim-backend/simulation_for_article/base_model

   calibration_report.json                           30 kB
   links_baseline.parquet                         2,580 kB
   mapa_baseline.html                             5,451 kB
   metrics.json                                       2 kB
   network_summary.json                             564 kB
   obrazky                                            0 kB
   od_summary.json                                    4 kB
   sim.yaml.snapshot                                  3 kB
   validation_report.json                            29 kB


In [15]:
# ═══ POROVNANIE DVOCH VARIANTOV ══════════════════════════════════════
# Spustiť až keď existujú oba priečinky.

A = Path("../simulation_for_article/base_model/metrics.json")
B = Path("../simulation_for_article/updated_version_1/metrics.json")

if A.exists() and B.exists():
    a, b = json.loads(A.read_text()), json.loads(B.read_text())

    riadky = []
    for sekcia, kluce in [
        ("dopyt", ["od_spolu", "commuting", "other", "external_local", "external_through"]),
        ("kalibracia", ["n", "r2", "slope", "pct_rmse", "bias_pct"]),
        ("holdout", ["r2", "slope", "pct_rmse", "bias_pct"]),
        ("odme", ["drift_od_seedu_pct", "buniek_nad_prahom", "Z_koniec"]),
        ("baseline_kpi", ["vht_voz_h", "vkt_voz_km", "pretazenych_hran"]),
    ]:
        for k in kluce:
            va = (a.get(sekcia) or {}).get(k)
            vb = (b.get(sekcia) or {}).get(k)
            if va is None and vb is None:
                continue
            try:
                zmena = round(vb - va, 4)
                zmena_pct = round((vb - va) / abs(va) * 100, 1) if va else None
            except Exception:
                zmena = zmena_pct = None
            riadky.append({"sekcia": sekcia, "metrika": k,
                           "base": va, "updated": vb,
                           "zmena": zmena, "zmena_%": zmena_pct})

    porovnanie = pd.DataFrame(riadky)
    display(porovnanie)
    porovnanie.to_csv("../simulation_for_article/porovnanie.csv", index=False)
    print("→ ../simulation_for_article/porovnanie.csv")
else:
    print("Porovnanie preskočené — chýba jeden z variantov:")
    print(f"  base_model        : {'✓' if A.exists() else '✗'}")
    print(f"  updated_version_1 : {'✓' if B.exists() else '✗'}")

Porovnanie preskočené — chýba jeden z variantov:
  base_model        : ✓
  updated_version_1 : ✗
